In [1]:
import os
import nltk
import spacy
import string
import mlflow
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://fsn1.your-objectstorage.com"
os.environ["AWS_ACCESS_KEY_ID"] = "KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "ACCESS_KEY"

In [3]:
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri("http://localhost:5000")

mlflow.autolog()

if not mlflow.get_experiment_by_name(name="TF_IDF Model"):
    print("Creating model...")
    mlflow.create_experiment(name="TF_IDF Model")


if not mlflow.get_experiment_by_name(name="Count Model"):
    print("Creating model...")
    mlflow.create_experiment(name="Count Model", artifact_location="s3://mlflowartifacts/count_model")


tfidf_experiment = mlflow.get_experiment_by_name(name="TF_IDF Model")
count_experiment = mlflow.get_experiment_by_name(name="Count Model")

2026/05/15 04:24:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


Creating model...
Creating model...


In [4]:
IMDB_DATA_PATH = "data/raw/imdb.csv"

# Pre Processing

In [5]:
def clean_text(text: str) -> str:
    """
    Do some text cleaning and tokenizing
    """

    text = str(text).lower()
    text = text.replace("<br />", " ") # Remove html tags
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    doc = nlp(text)
    cleaned_tokens = [
        token.lemma_ for token in doc
        if token.text not in stop_words and token.is_alpha
    ]

    return " ".join(cleaned_tokens)

In [6]:
# Download stopwords
if not os.path.exists(IMDB_DATA_PATH):
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

    nlp = spacy.load("en_core_web_sm")

    df = pd.read_csv("https://iai.fsn1.your-objectstorage.com/imdb_dataset.csv")

    tqdm.pandas(desc='Cleaning Text...') 
    df['review_clean'] = df['review'].progress_apply(clean_text)

    df.to_csv(IMDB_DATA_PATH, columns=['sentiment', 'review_clean'], index=False)

# Model Training

In [7]:
df_tfidf = pd.read_csv(IMDB_DATA_PATH)

In [8]:
# Map string to int
df_tfidf['sentiment'] = df_tfidf['sentiment'].map({'positive': 1, 'negative': 0})

In [9]:
dataset = load_dataset("imdb", cache_dir="data/cache")

In [10]:
vecorizer = CountVectorizer()
tfidf = TfidfVectorizer()

In [11]:
data_count = vecorizer.fit_transform(dataset['train']['text'])
data_tfidf = tfidf.fit_transform(df_tfidf['review_clean'])

In [12]:
x_train_count, x_test_count, y_train_count, y_test_count = train_test_split(data_count, dataset['train']['label'], test_size=0.99, random_state=42)
x_train_tfidf, x_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(data_tfidf, df_tfidf['sentiment'].values, test_size=0.2, random_state=42)

In [13]:
with mlflow.start_run(experiment_id=count_experiment.experiment_id):
    model_count = LogisticRegression(random_state=42, max_iter=1000)
    model_count.fit(x_train_count, np.array(y_train_count))

    count_run = mlflow.active_run().info

2026/05/15 04:24:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/15 04:24:35 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/05/15 04:24:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


🏃 View run rumbling-midge-361 at: http://localhost:5000/#/experiments/2/runs/3eb094b7a2cf4c9ca6a22d74bd471aee
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [ ]:
with mlflow.start_run(experiment_id=tfidf_experiment.experiment_id):
    model_tfidf = LogisticRegression(random_state=42, max_iter=1000)
    model_tfidf.fit(x_train_tfidf, y_train_tfidf)

    tfidf_run = mlflow.active_run().info

In [14]:
prediction_count = model_count.predict(x_test_count)

scores_count = np.array([[
    accuracy_score(y_test_count, prediction_count),
    recall_score(y_test_count, prediction_count),
    precision_score(y_test_count, prediction_count),
]])

print(f"{'Accuracy_score':<18} {scores_count[0][0]:.4f}")
print(f"{'Recall_score':<18} {scores_count[0][1]:.4f}")
print(f"{'Precision_score':<18} {scores_count[0][1]:.4f}")

mlflow.log_metrics(run_id=count_run.run_id, metrics={
    "testing_accuracy_score": accuracy_score(y_test_count, prediction_count),
    "testing_recall_score": recall_score(y_test_count, prediction_count),
    "testing_precision_score": precision_score(y_test_count, prediction_count),
})


Accuracy_score     0.7300
Recall_score       0.7415
Precision_score    0.7415


In [ ]:
prediction_tfidf = model_tfidf.predict(x_test_tfidf)

scores_tfidf = np.array([[
    accuracy_score(y_test_tfidf, prediction_tfidf),
    recall_score(y_test_tfidf, prediction_tfidf),
    precision_score(y_test_tfidf, prediction_tfidf),
]])

print(f"{'Accuracy_score':<18} {scores_tfidf[0][0]:.4f}")
print(f"{'Recall_score':<18} {scores_tfidf[0][1]:.4f}")
print(f"{'Precision_score':<18} {scores_tfidf[0][2]:.4f}")

mlflow.log_metrics(run_id=tfidf.run_id, metrics={
    "testing_accuracy_score": accuracy_score(y_test_tfidf, prediction_tfidf),
    "testing_recall_score": recall_score(y_test_tfidf, prediction_tfidf),
    "testing_precision_score": precision_score(y_test_tfidf, prediction_tfidf),
})

Accuracy_score     0.8918
Recall_score       0.9065
Precision_score    0.8820


In [ ]:
scores_delta = scores_tfidf - scores_count

print("Delta (TF_IDF vs Count) \n")

print(f"{'Accuracy_score':<18} {scores_delta[0][0]:.4f}")
print(f"{'Recall_score':<18} {scores_delta[0][1]:.4f}")
print(f"{'Precision_score':<18} {scores_delta[0][2]:.4f}")

Delta (TF_IDF vs Count) 

Accuracy_score     0.0114
Recall_score       0.0140
Precision_score    0.0119


In [ ]:
eval_df = pd.DataFrame(x_test_count)
eval_df["label"] = y_test_count

mlflow.models.evaluate(
    model="s3://mlflowartifacts/count_model/models/m-2fa0f953c5014ff79b2989206edf599a/artifacts/",
    model_id="m-2fa0f953c5014ff79b2989206edf599a",
    model_type="classifier",
    targets="label",
    data=eval_df
)

MlflowException: The extra_metrics argument must be specified model_type is None.